## 주요 변경점 및 결과
1. Model은 EfficientNet만 사용하되, Simulation으로 생성한 영상 데이터를 추가해 Video Feature Extraction + Distillation Weight 변동
2. Simulation으로 생성한 데이터로는 thin tower / thick tower / wall / pyramid 4개만 우선 사용
3. 원본 Train Data의 비디오에서 프레임을 추출하는 로직은 주석 처리하고 실험

In [1]:
import os
os.environ['CUDA_MPS_PIPE_DIRECTORY'] = '/tmp/nvidia-mps'
os.environ['CUDA_MPS_LOG_DIRECTORY'] = '/tmp/nvidia-mps-log'

## 1. 라이브러리 및 환경 설정

In [2]:
import os
import sys
import json
import random
import pandas as pd
import numpy as np
import cv2
import shutil
import timm
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from pathlib import Path

ROOT = Path.cwd().resolve().parent
SRC_DIR = (Path.cwd() / '../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker

# /src 에서 실행하는 기준 경로 설정
DATA_DIR = (Path.cwd() / '../data').resolve()
SIMULATION_DATA_DIR = (Path.cwd() / '../tools/data/generated_v2_0').resolve()
EXPERIMENT_NAME = "Data_AUG_Test"
EXPERIMENT_VERSION = "v3.1"
WEIGHT_DIR = ROOT / "outputs" / "weights" / EXPERIMENT_NAME / EXPERIMENT_VERSION
SUBMISSION_DIR = ROOT / "outputs" / "submissions" / EXPERIMENT_NAME / EXPERIMENT_VERSION
EDA_DIR = ROOT / "outputs" / "eda_preprocessing" / EXPERIMENT_NAME / EXPERIMENT_VERSION
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"data 폴더를 찾지 못했습니다: {DATA_DIR}"
print(f"DATA_DIR: {DATA_DIR}")

# 하이퍼파라미터 설정
CFG = {
    # Basic
    'IMG_SIZE': 320,
    'EPOCHS': 30,
    'LEARNING_RATE': 3e-4,
    'MIN_LR': 1e-6,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'NUM_WORKERS': 16,
    'USE_AMP' : True,
    # Regularization & Stability
    'WEIGHT_DECAY': 1e-4,  # L2 regularization
    'EMA_DECAY': 0.999, # 시계열에서 window size만큼 고려해 지역적 평균 구하는 방식으로 노이즈를 제거
    'EMA_USE_FOR_EVAL': True,
    'PATIENCE' : 10,
    # Augmentation & TTA
    'TTA_CANDIDATES': [ # TEST TIME AUGMENTATION
        ['none'],
        ['none', 'hflip'],
        ['none', 'hflip', 'crop95'],
    ],
    'DISTILL_WEIGHT': [1e-4, 1e-3, 1e-2, 1e-1], # Distill Weight for Knowledge Distillation
    # video frame augmentation (for unstable videos)
    'VIDEO_AUG_ENABLE': True,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
}

BACKBONE_CANDIDATES = [
    "efficientnetv2_rw_s",
]

selected_backbones = []
for name in BACKBONE_CANDIDATES:
    try:
        timm.create_model(name, pretrained=False, num_classes=0, global_pool="")
        selected_backbones.append(name)
    except Exception as exc:
        print("skip backbone:", name, exc)

print("selected_backbones:", selected_backbones)
assert selected_backbones, "No candidate backbones are available in this timm installation."

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DATA_DIR: /home/vsc/LLM_TUNE/structure-stability/data
selected_backbones: ['efficientnetv2_rw_s']
cuda


## 2. 데이터 로드 및 학습/검증 데이터 분할

In [3]:
# 1. 데이터 로드
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
val_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

학습 데이터 개수: 1000
검증 데이터 개수: 100


## 3. 영상에서 Feature을 추출하는 Teacher 모델 선언

In [4]:
from transformers import VideoMAEImageProcessor, VideoMAEModel

# ────────────────────────────────────────────────────────
# 설정
# ────────────────────────────────────────────────────────
DATA_DIR     = Path('../data').resolve()
TRAIN_ROOT   = DATA_DIR / 'train'
SIMULATION_TRAIN_ROOT = SIMULATION_DATA_DIR  
FEATURE_DIR  = DATA_DIR / 'videomae_features'   # feature 저장 경로
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME   = '/home/vsc/LLM/model/videomae-base-finetuned-kinetics'
NUM_FRAMES   = 16       # VideoMAE 입력 프레임 수 (고정)
FRAME_SIZE   = 224      # VideoMAE 입력 해상도 (고정)
BATCH_SIZE   = 32        # 한 번에 처리할 비디오 수 (VRAM에 따라 조정)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")
print(f"Output : {FEATURE_DIR}")

# ────────────────────────────────────────────────────────
# 모델 로드
# ────────────────────────────────────────────────────────
print("\n모델 로딩 중...")
processor = VideoMAEImageProcessor.from_pretrained(MODEL_NAME) # MAE : Masked Auto Encoders
model = VideoMAEModel.from_pretrained(MODEL_NAME)
model.eval()
model.to(DEVICE)
print("모델 로딩 완료")

# ────────────────────────────────────────────────────────
# 비디오 → 프레임 추출 함수
# ────────────────────────────────────────────────────────
def load_video_frames(video_path: Path, num_frames: int = 16) -> list | None:
    """
    비디오에서 균등 간격으로 num_frames개의 프레임을 추출
 
    Returns:
        list of np.ndarray (H, W, 3), RGB / None if failed
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
 
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return None
 
    # 균등 간격으로 num_frames개 인덱스 선택
    # ex) frame_count=300, num_frames=16 → [0, 20, 40, ..., 300]
    indices = np.linspace(0, frame_count - 1, num_frames, dtype=int)
 
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            # 실패 시 마지막 성공 프레임으로 대체
            if frames:
                frames.append(frames[-1].copy())
            else:
                cap.release()
                return None
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)
 
    cap.release()
 
    if len(frames) != num_frames:
        return None
 
    return frames  # list of (H, W, 3) np.ndarray

# ────────────────────────────────────────────────────────
# feature 추출 함수
# ────────────────────────────────────────────────────────


@torch.no_grad()
def extract_features(frames: list) -> np.ndarray:
    inputs = processor(frames, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    outputs = model(**inputs)
    cls_feature = outputs.last_hidden_state[:, 0, :]  # (1, 768)
 
    return cls_feature.squeeze(0).cpu().numpy()        # (768,)

# ────────────────────────────────────────────────────────
# 전체 데이터셋에 대해 feature 추출 및 저장
# ────────────────────────────────────────────────────────
def extract_all_features(train_df: pd.DataFrame):
    """
    모든 샘플에 대해 feature 추출 후 .npy로 저장
 
    저장 구조:
        data/videomae_features/
            000001.npy   ← shape (768,)
            000002.npy
            ...
        data/videomae_features/feature_meta.json  ← 성공/실패 기록
    """
    meta = {}
    failed = []
 
    for _, row in tqdm(train_df.iterrows(),
                       total=len(train_df),
                       desc='VideoMAE feature 추출',
                       dynamic_ncols=True,
                       ascii=True):
 
        sample_id  = str(row['id'])
        sample_dir = row.get('sample_dir', TRAIN_ROOT)
        video_path = Path(sample_dir) / sample_id / 'simulation.mp4'
        save_path  = FEATURE_DIR / f'{sample_id}.npy'
 
        # 이미 추출된 경우 스킵
        if save_path.exists():
            meta[sample_id] = 'cached'
            continue
 
        if not video_path.exists():
            meta[sample_id] = 'no_video'
            failed.append(sample_id)
            continue
 
        # 프레임 추출
        frames = load_video_frames(video_path, num_frames=NUM_FRAMES)
        if frames is None:
            meta[sample_id] = 'frame_extract_failed'
            failed.append(sample_id)
            continue
 
        # feature 추출
        try:
            feature = extract_features(frames)   # (768,)
            np.save(save_path, feature)
            meta[sample_id] = 'ok'
        except Exception as e:
            meta[sample_id] = f'error: {e}'
            failed.append(sample_id)
 
    # 메타 저장
    meta_path = FEATURE_DIR / 'feature_meta.json'
    meta_path.write_text(
        json.dumps(meta, ensure_ascii=False, indent=2)
    )
 
    print(f"\n완료: {len(train_df) - len(failed)} / {len(train_df)} 성공")
    if failed:
        print(f"실패 샘플 ({len(failed)}개): {failed[:10]}{'...' if len(failed) > 10 else ''}")
 
    return meta
 
# ────────────────────────────────────────────────────────
# feature 로드 유틸 (학습 코드에서 사용)
# ────────────────────────────────────────────────────────
def load_feature(sample_id: str) -> np.ndarray | None:
    """
    저장된 feature 로드
 
    Usage:
        feat = load_feature('000001')  # shape (768,)
    """
    path = FEATURE_DIR / f'{sample_id}.npy'
    if not path.exists():
        return None
    return np.load(path)
 
 
def load_all_features(sample_ids: list) -> dict:
    """
    여러 샘플의 feature를 한 번에 로드
 
    Returns:
        dict: {sample_id: np.ndarray(768,)} 또는 None(실패)
    """
    features = {}
    for sid in sample_ids:
        features[sid] = load_feature(sid)
    return features

Device : cuda
Model  : /home/vsc/LLM/model/videomae-base-finetuned-kinetics
Output : /home/vsc/LLM_TUNE/structure-stability/data/videomae_features

모델 로딩 중...


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: /home/vsc/LLM/model/videomae-base-finetuned-kinetics
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
fc_norm.weight    | UNEXPECTED |  | 
fc_norm.bias      | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


모델 로딩 완료


In [5]:
from PIL import Image
from pathlib import Path

# 1. 샘플 이미지 경로 하나 지정 (앞서 사용하신 경로 기준)
sample_path1 = Path(DATA_DIR) / "train" / "TRAIN_0010" / "front.png" 
sample_path2 = Path(SIMULATION_DATA_DIR) / "SIM20_0001" / "front.png" 

# 2. 이미지 열기
img1 = Image.open(sample_path1)
img2 = Image.open(sample_path2)

# 3. 원본 사이즈 확인 (가로, 세로)
width1, height1 = img1.size
print(f"원본 이미지 사이즈: {width1} x {height1}")

width2, height2 = img2.size
print(f"Simulation AUG 이미지 사이즈: {width2} x {height2}")

원본 이미지 사이즈: 384 x 384
Simulation AUG 이미지 사이즈: 384 x 384


crop_size : 224*224에서
patch_size :16으로, 16 * 16 크기의 작은 타일로 쪼개고

가로 & 세로 : 224 / 16 = 14개
총 조각 수 : 14 * 14 = 196개

최종 정보량 : 패치 내 픽셀 수 16 * 16 * 3(RGB) = 768

In [6]:
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
print(f"학습 데이터: {len(train_df)}개")

meta = extract_all_features(train_df)

# 샘플 확인
sample_id = str(train_df.iloc[0]['id'])
feat = load_feature(sample_id)
if feat is not None:
    print(f"\n샘플 feature shape : {feat.shape}")   # (768,)
    print(f"샘플 feature 통계  : mean={feat.mean():.4f}, std={feat.std():.4f}")

학습 데이터: 1000개


VideoMAE feature 추출: 100%|##########| 1000/1000 [00:00<00:00, 33300.81it/s]


완료: 1000 / 1000 성공

샘플 feature shape : (768,)
샘플 feature 통계  : mean=0.2648, std=7.8716


In [7]:
import json
import pandas as pd
from pathlib import Path
sim_rows = []
print(f"시뮬레이션 데이터 폴더를 탐색합니다: {SIMULATION_DATA_DIR}")

# SIMULATION_DATA_DIR 내의 SIM20_ 으로 시작하는 모든 폴더를 순회
for sim_dir in SIMULATION_DATA_DIR.glob('SIM20_*'):
    if sim_dir.is_dir():
        meta_path = sim_dir / 'meta.json'
        if meta_path.exists():
            with open(meta_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
                
            sim_rows.append({
                'id': meta['id'], 
                'label': meta['label'], 
                'structure': meta['structure_requested'],
                'sample_dir': str(SIMULATION_DATA_DIR)
            })

# DataFrame으로 만들고 Feature 추출 진행
if sim_rows:
    sim_df = pd.DataFrame(sim_rows)
    display(sim_df.head(10))
    print(f"\n로드된 시뮬레이션 데이터: {len(sim_df)}개")

시뮬레이션 데이터 폴더를 탐색합니다: /home/vsc/LLM_TUNE/structure-stability/tools/data/generated_v2_0


,id,label,structure,sample_dir
0,SIM20_0004,unstable,thin tower,/home/vsc/LLM_TUNE/structure-stability/tools/d...
1,SIM20_0458,unstable,thin triangle,/home/vsc/LLM_TUNE/structure-stability/tools/d...
2,SIM20_0125,unstable,thin tower,/home/vsc/LLM_TUNE/structure-stability/tools/d...
3,SIM20_0152,unstable,thin tower,/home/vsc/LLM_TUNE/structure-stability/tools/d...
4,SIM20_0267,stable,wall,/home/vsc/LLM_TUNE/structure-stability/tools/d...
5,SIM20_0203,stable,thick tower,/home/vsc/LLM_TUNE/structure-stability/tools/d...
6,SIM20_0142,stable,thin tower,/home/vsc/LLM_TUNE/structure-stability/tools/d...
7,SIM20_0439,stable,thin triangle,/home/vsc/LLM_TUNE/structure-stability/tools/d...
8,SIM20_0280,unstable,wall,/home/vsc/LLM_TUNE/structure-stability/tools/d...
9,SIM20_0288,unstable,wall,/home/vsc/LLM_TUNE/structure-stability/tools/d...



로드된 시뮬레이션 데이터: 511개


In [8]:
target_structures = ['thin tower', 'thick tower', 'wall', 'pyramid']
selected_sim_df = sim_df[sim_df['structure'].isin(target_structures)].copy()
print((selected_sim_df['label'] == 'stable').sum())
print((selected_sim_df['label'] == 'unstable').sum())

184
216


In [9]:
if sim_rows:
    extract_all_features(selected_sim_df)
else :
    print("조건에 맞는 시뮬레이션 데이터를 찾지 못했습니다.")

VideoMAE feature 추출: 100%|##########| 400/400 [00:00<00:00, 37034.16it/s]


완료: 400 / 400 성공


## 4-1. CheckerboardTopNormalizer 클래스 & CheckerboardTopNormConfig 설정 정의

In [10]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional
from PIL import Image

import cv2
import numpy as np
import matplotlib.pyplot as plt

@dataclass(frozen=True)
class CheckerboardTopNormConfig:
    enabled: bool = True
    ring_ratio: float = 0.10
    rot_line_min: int = 10
    rot_conf_min: float = 0.20
    pad_value: int = 128


def _estimate_mask(rgb: np.ndarray) -> np.ndarray:
    """
    지저분한 마스크 다듬기 & 원하는 물체만 골라내기
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) 
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV) 
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    s_thr = float(np.percentile(sat, 60.0))
    g_thr = float(np.percentile(gray, 45.0))
    v_thr = float(np.percentile(val, 35.0))

    mask = ((sat > s_thr) | (gray < g_thr) | (val < v_thr)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=2)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return (mask > 0).astype(np.uint8)

    best = 1
    best_area = 0
    h, w = gray.shape
    for i in range(1, n):
        x, y, ww, hh, area = stats[i].tolist()
        if area > best_area and ww > 8 and hh > 8 and area < 0.995 * h * w:
            best = i
            best_area = area
    return (labels == best).astype(np.uint8)


def _ring_mask(h: int, w: int, ratio: float) -> np.ndarray:
    r = max(1, int(round(min(h, w) * ratio)))
    mask = np.zeros((h, w), dtype=np.uint8)
    mask[:r, :] = 1
    mask[-r:, :] = 1
    mask[:, :r] = 1
    mask[:, -r:] = 1
    return mask


def _line_angles(edge: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    추출한 Edge 이미지에서 직선을 찾고, 기울어진 각을 계산해 반환하는 함수
    """
    lines = cv2.HoughLinesP(edge, 1, np.pi / 180.0, threshold=30, minLineLength=24, maxLineGap=6)
    if lines is None or len(lines) == 0:
        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32)

    angs = []
    lens = []
    for line in lines[:400]:
        x1, y1, x2, y2 = line[0].tolist()
        dx = float(x2 - x1)
        dy = float(y2 - y1)
        ln = float(np.hypot(dx, dy))
        if ln < 8:
            continue
        ang = (float(np.degrees(np.arctan2(dy, dx))) + 180.0) % 180.0
        angs.append(ang)
        lens.append(ln)
    if len(angs) == 0:
        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32)
    return np.asarray(angs, dtype=np.float32), np.asarray(lens, dtype=np.float32)


def estimate_top_rotation(rgb: np.ndarray, cfg: CheckerboardTopNormConfig) -> dict[str, float | bool | int | str]:
    h, w = rgb.shape[:2]
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    fg = _estimate_mask(rgb) # 원하는 물체를 추출하는 함수
    bg = (1 - fg).astype(np.uint8)
    if int(bg.sum()) < int(0.03 * h * w):
        bg = _ring_mask(h, w, cfg.ring_ratio)

    gx = cv2.Scharr(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(gray, cv2.CV_32F, 0, 1)
    mag = cv2.magnitude(gx, gy)
    mag = (mag / (mag.max() + 1e-6) * 255.0).astype(np.uint8)
    edges = cv2.Canny(mag, 40, 120)
    edges = cv2.bitwise_and(edges, edges, mask=(bg * 255))

    angles, lengths = _line_angles(edges)
    line_n = int(len(angles))
    if line_n == 0:
        return {
            "angle_deg": 0.0,
            "rot_conf": 0.0,
            "rot_ok": False,
            "rot_fail_reason": "no_lines",
            "rot_line_count": 0,
        }

    hist, _ = np.histogram(angles, bins=180, range=(0.0, 180.0), weights=lengths)
    peak_primary = int(np.argmax(hist))
    peak_primary_value = float(hist[peak_primary])

    hist_secondary = hist.copy()
    for d in range(-8, 9):
        hist_secondary[(peak_primary + d) % 180] = 0.0
    peak_secondary = int(np.argmax(hist_secondary))
    peak_secondary_value = float(hist_secondary[peak_secondary])
    peak_orthogonal = float(hist[(peak_primary + 90) % 180])

    mods = np.mod(angles, 90.0)
    theta = mods * (2.0 * np.pi / 90.0)
    cx = float(np.sum(lengths * np.cos(theta)))
    cy = float(np.sum(lengths * np.sin(theta)))
    rot_mod90 = float((np.degrees(np.arctan2(cy, cx)) * (90.0 / 360.0)) % 90.0)

    peak_ratio_score = peak_primary_value / (peak_primary_value + peak_secondary_value + 1e-6)
    line_score = min(1.0, line_n / 60.0)
    spread = float(np.sqrt(np.average((mods - np.average(mods, weights=lengths)) ** 2, weights=lengths)))
    spread_score = float(np.clip(1.0 - spread / 20.0, 0.0, 1.0))
    ortho_score = float(np.clip(peak_orthogonal / (peak_primary_value + 1e-6), 0.0, 1.0))
    conf = float(np.clip(0.35 * peak_ratio_score + 0.25 * line_score + 0.20 * spread_score + 0.20 * ortho_score, 0.0, 1.0))

    reasons = []
    if line_n < cfg.rot_line_min:
        reasons.append("line_count_low")
    if conf < cfg.rot_conf_min:
        reasons.append("confidence_low")

    return {
        "angle_deg": -rot_mod90,
        "rot_conf": conf,
        "rot_ok": len(reasons) == 0,
        "rot_fail_reason": "|".join(reasons),
        "rot_line_count": line_n,
    }

def rotate_rgb(rgb: np.ndarray, angle_deg: float, pad_value: int) -> np.ndarray:
    if abs(angle_deg) < 1e-6:
        return rgb
    h, w = rgb.shape[:2]
    matrix = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle_deg, 1.0)
    return cv2.warpAffine(
        rgb,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(pad_value, pad_value, pad_value),
    )


class CheckerboardTopNormalizer:
    def __init__(self, cfg: Optional[CheckerboardTopNormConfig] = None) -> None:
        self.cfg = cfg or CheckerboardTopNormConfig()
        self._angle_cache: Dict[str, Optional[float]] = {}

    def normalize(self, path: str | Path, image: Image.Image, debug_save_path: Optional[Path] = None) -> Image.Image:
            """
            이미지를 정규화(회전 보정)합니다.
            debug_save_path가 전달되면 중간 분석 과정을 시각화하여 저장합니다.
            """
            if not self.cfg.enabled:
                return image

            key = str(Path(path).expanduser().resolve())
            
            # 1. 각도 계산 및 캐싱 (이미 계산된 적이 없다면 실행)
            if key not in self._angle_cache:
                rgb = np.asarray(image.convert("RGB"))
                info = estimate_top_rotation(rgb, self.cfg)
                
                # 성공 여부에 따라 각도 저장 (실패 시 None)
                self._angle_cache[key] = float(info["angle_deg"]) if bool(info["rot_ok"]) else None

                # [핵심 수정] 파라미터로 받은 debug_save_path가 있을 때만 시각화 함수 호출
                if debug_save_path:
                    self._save_debug_plot(rgb, info, debug_save_path)

            # 2. 캐시된 각도 가져오기
            angle = self._angle_cache[key]
            
            # 보정할 각도가 없거나(None), 실패한 경우 원본 반환
            if angle is None:
                return image

            # 3. 실제 회전 적용
            rgb = np.asarray(image.convert("RGB"))
            rotated = rotate_rgb(rgb, angle_deg=angle, pad_value=self.cfg.pad_value)
            
            return Image.fromarray(rotated)
    
    def _save_debug_plot(self, rgb: np.ndarray, info: dict, save_path: Path):
        # 중간 과정 재현 (시각화용)
        fg_mask = _estimate_mask(rgb)
        bg_mask = (1 - fg_mask).astype(np.uint8)
        if int(bg_mask.sum()) < int(0.03 * rgb.shape[0] * rgb.shape[1]):
            bg_mask = _ring_mask(rgb.shape[0], rgb.shape[1], self.cfg.ring_ratio)
        
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 40, 120) # 단순화를 위해 바로 Canny 적용 가능
        masked_edges = cv2.bitwise_and(edges, edges, mask=(bg_mask * 255))
        
        line_img = rgb.copy()
        lines = cv2.HoughLinesP(masked_edges, 1, np.pi / 180.0, threshold=30, minLineLength=24, maxLineGap=6)
        if lines is not None:
            for line in lines[:100]:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        rotated = rotate_rgb(rgb, info["angle_deg"], self.cfg.pad_value)

        # 플롯 생성
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        titles = ["Original", "Edges", "Lines", "Result"]
        imgs = [rgb, masked_edges, line_img, rotated]
        
        for ax, img, title in zip(axes, imgs, titles):
            ax.imshow(img, cmap='gray' if len(img.shape) == 2 else None)
            ax.set_title(title)
            ax.axis('off')
        
        fig.suptitle(f"Angle: {info['angle_deg']:.2f}° | Conf: {info['rot_conf']:.2f}", fontsize=15)
        
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight')
        plt.close(fig)

## 4-2. VideoMAE & CheckerboardTopNormalization 을 포함한 커스텀 클래스 + 데이터 로더 위한 함수 정의

In [11]:
class MultiViewDataset(Dataset):
    def __init__(
        self,
        df,
        root_dir,
        transform=None, # 이 transform이 Compose라면 내부 시점별 분기가 필요할 수 있음
        is_test=False,
        feature_dir=None,
        checkerboard_top_normalize: bool = True
    ):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}
        self.feature_dir = Path(feature_dir) if feature_dir else None
        
        # 체커보드 정규화기 설정
        self.top_normalizer = CheckerboardTopNormalizer(
            CheckerboardTopNormConfig(enabled=True)
        ) if checkerboard_top_normalize else None

    def __len__(self):
        return len(self.df)

    def _load_img(self, sid: str, view: str, sample_dir: str) -> Image.Image:
        # 비디오 증강 데이터 여부 확인 (ID prefix 사용)
        is_augv = sid.startswith('AUGV_')
        
        # 경로 설정
        path = Path(sample_dir) / sid / f"{view}.png"
        image = Image.open(path).convert("RGB")
        
        # [핵심 수정] Top View 정규화 로직
        # 비디오 증강 데이터(AUGV_)가 아닐 때만 체커보드 정규화 수행
        if view == "top" and self.top_normalizer is not None and not is_augv:
            debug_path = None
            # 학습/검증 중 앞부분만 디버깅 이미지 저장
            if not self.is_test and len(self.top_normalizer._angle_cache) < 20:
                folder = "train" if "TRAIN" in sid else "dev"
                debug_path = Path(f"./debug_plots/{folder}/{sid}_top.png")

            image = self.top_normalizer.normalize(path, image, debug_save_path=debug_path)
            
        return image

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = str(row['id'])
        # sample_dir이 row에 있으면 사용, 없으면 기본 root_dir 사용
        sample_dir = row['sample_dir'] if 'sample_dir' in row else self.root_dir

        views = []
        for name in ['front', 'top']:
            # 정규화 로직이 포함된 이미지 로드 호출
            image = self._load_img(sid, name, sample_dir)
            
            # transform 적용
            if self.transform:
                # 만약 transform이 시점별로 다르다면(Compose 내부 분기 등) 여기서 처리
                image = self.transform(image)
            
            views.append(image)

        # VideoMAE feature 로드 (기존 로직 유지)
        feature = torch.zeros(768)
        if self.feature_dir is not None:
            feat_path = self.feature_dir / f'{sid}.npy'
            if feat_path.exists():
                feature = torch.from_numpy(np.load(feat_path)).float()
            elif sid.startswith('AUGV_'):
                # 증강 데이터인데 피처가 없다면? 
                # 원본 비디오의 피처를 찾거나(복잡), 그냥 0으로 유지
                pass

        if self.is_test:
            return views, feature

        label = self.label_map[row['label']]
        return views, label, feature

In [12]:
def _extract_frame_by_sec(cap, sec, fps, frame_count):
    # 매 프레임에 해당하는 장면을 가져오는 함수
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    # 가장 마지막 프레임을 가져오는 함수
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    # VIDEO_AUG에 해당하는 CFG만 가져오는 함수
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}

def detect_collapse_start(cap, fps, frame_count, motion_threshold=0.02):
    """
    프레임 간 픽셀 변화량으로 붕괴 시작 시점 탐지
    """
    prev_frame = None
    collapse_sec = None

    # 전체 영상을 일정 간격으로 샘플링
    sample_interval = max(1, int(fps * 0.2))  # 0.2초 간격

    for frame_idx in range(0, frame_count, sample_interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(float) / 255.0

        if prev_frame is not None:
            # 프레임 간 변화량 계산
            diff = np.mean(np.abs(gray - prev_frame))
            if diff > motion_threshold:
                collapse_sec = frame_idx / fps
                break  # 첫 번째 큰 변화 시점

        prev_frame = gray

    return collapse_sec  # None이면 붕괴 미감지

def build_video_augmented_df(train_df, data_dir, cfg):
    """
    train의 simulation.mp4에서 프레임 추출 (train의 경우 )
    stable : 그냥 계속 동일한 프레임들이 나올테니, 그냥 마지막 프레임 뽑음
    unstable : 0~10초 내에 구조물이 무너지는 순간이 있을텐데, 이 변화량이 큰 순간을 포착
    """
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    print(f'video aug cache hit: {len(cached_df)} samples from {cache_csv}')
                    return cached_df
        except Exception as e:
            print(f'video aug cache read failed. rebuild cache. ({e})')

    # 캐시 미스 시 기존 AUGV_* 폴더만 정리 후 재생성
    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    stable_rows = []
    unstable_rows = []
    saved_idx = 0

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        row = {'id': aug_id, 'label': label, 'sample_dir': str(aug_root)}
        saved_idx += 1
        return row

    # 1) stable 증강: 영상의 마지막 프레임 1장씩 사용
    all_ids = train_df['id'].tolist()
    for sample_id in tqdm(all_ids, desc='Video aug stable(last-frame)', dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            continue

        img = _extract_last_frame(cap, frame_count)
        cap.release()
        if img is None:
            continue

        stable_rows.append(save_aug(img, 'stable'))

    # 2) unstable 증강: 실제로 해당 구조물이 붕괴하기 시작하는 시점을 탐지
    unstable_ids = train_df.loc[train_df['label'] == 'unstable', 'id'].tolist()
    for sample_id in tqdm(unstable_ids, desc='Video aug unstable(collapse-detect)', dynamic_ncols=True, ascii=True):
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        collapse_sec = detect_collapse_start(cap, fps, frame_count)

        if collapse_sec is None:
            low  = cfg['UNSTABLE_START_MIN_SEC']
            high = cfg['UNSTABLE_START_MAX_SEC']
        else:
            low  = max(0.0, collapse_sec - 0.3)   # 붕괴 0.3초 전
            high = min(collapse_sec + 0.5,          # 붕괴 0.5초 후
                       frame_count / fps)

        n_unstable = int(rng.integers(
            cfg['UNSTABLE_FRAMES_MIN'],
            cfg['UNSTABLE_FRAMES_MAX'] + 1
        ))
        unstable_secs = rng.uniform(low, high, size=n_unstable)

        for sec in unstable_secs:
            img = _extract_frame_by_sec(cap, sec, fps, frame_count)
            if img is None:
                continue
            unstable_rows.append(save_aug(img, 'unstable'))

        cap.release()

    stable_df = pd.DataFrame(stable_rows)
    unstable_df = pd.DataFrame(unstable_rows)

    if stable_df.empty or unstable_df.empty:
        print('video aug warning: stable/unstable 중 하나가 비어 비율 매칭 불가')
        return pd.DataFrame(columns=['id', 'label', 'sample_dir'])

    # 3) stable 개수에 맞춰 unstable 개수 정렬
    target_unstable = len(stable_df)
    if len(unstable_df) >= target_unstable:
        unstable_bal = unstable_df.sample(n=target_unstable, random_state=cfg['SEED'])
    else:
        unstable_bal = unstable_df.sample(n=target_unstable, replace=True, random_state=cfg['SEED'])

    aug_df = pd.concat([stable_df, unstable_bal], ignore_index=True)
    aug_df = aug_df.sample(frac=1.0, random_state=cfg['SEED']).reset_index(drop=True)

    # 캐시 저장
    if cfg.get('VIDEO_AUG_CACHE', True):
        aug_df.to_csv(cache_csv, index=False)
        cache_meta.write_text(json.dumps({'signature': cache_sig}, ensure_ascii=False, indent=2))
        print(f'video aug cache saved: {cache_csv}')

    print(f'video aug stable(last-frame): {len(stable_df)}')
    print(f'video aug unstable(sampled): {len(unstable_bal)}')
    return aug_df

In [13]:
from sklearn.model_selection import train_test_split

train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])

# 원본 학습 데이터(기본 1:1)
train_df_copy = train_df.copy()
train_df_copy['sample_dir'] = str(DATA_DIR / 'train')

print(f"Before Augmentation || {train_df_copy['label'].value_counts()}")

# # 비디오 프레임 기반 증강 데이터 생성
# if CFG.get('VIDEO_AUG_ENABLE', False):
#     aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
#     if len(aug_df) > 0:
#         train_df_copy = pd.concat([train_df_copy, aug_df], ignore_index=True)
#         print(f'video aug added: {len(aug_df)} samples')
#     else:
#         print('video aug added: 0 samples (check video availability)')

# print(f"After Video Feature Extraction Augmentation || {train_df_copy['label'].value_counts()}")

sim_rows = []
for sim_dir in SIMULATION_DATA_DIR.glob('SIM20_*'):
    if sim_dir.is_dir():
        meta_path = sim_dir / 'meta.json'
        if meta_path.exists():
            import json
            with open(meta_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
            sim_rows.append({'id': meta['id'], 'label': meta['label'], 'sample_dir': str(SIMULATION_DATA_DIR)})

if sim_rows:
    sim_df = pd.DataFrame(sim_rows)
    
    stable_sim = sim_df[sim_df['label'] == 'stable']
    unstable_sim = sim_df[sim_df['label'] == 'unstable']
    min_count = min(len(stable_sim), len(unstable_sim))
    
    stable_bal = stable_sim.sample(n=min_count, random_state=CFG['SEED'])
    unstable_bal = unstable_sim.sample(n=min_count, random_state=CFG['SEED'])
    
    sim_df_balanced = pd.concat([stable_bal, unstable_bal], ignore_index=True)
    
    # --- [수정] Train / Dev 분할 (8:2) ---
    sim_train, sim_val = train_test_split(
        sim_df_balanced, 
        test_size=0.2, 
        random_state=CFG['SEED'], 
        stratify=sim_df_balanced['label'] # 양쪽 클래스 비율 1:1 유지
    )
    
    # Train에는 sim_train만 합치기
    train_df_copy = pd.concat([train_df_copy, sim_train], ignore_index=True)
    train_df_copy = train_df_copy.sample(frac=1.0, random_state=CFG['SEED']).reset_index(drop=True)
    
    print(f'Simulation data splitted - Train: {len(sim_train)} / Val: {len(sim_val)}')
    print(f'Total simulation data added to Train: {len(sim_train)} samples')
    
print('Final train class ratio:')
print(train_df_copy['label'].value_counts())


Before Augmentation || label
unstable    500
stable      500
Name: count, dtype: int64
Simulation data splitted - Train: 384 / Val: 96
Total simulation data added to Train: 384 samples
Final train class ratio:
label
stable      692
unstable    692
Name: count, dtype: int64


In [25]:
val_df_for_eval = val_df.copy()
val_df_for_eval['sample_dir'] = str(DATA_DIR / 'dev')

if 'sim_val' in globals() and not sim_val.empty:
    val_df_for_eval = pd.concat([val_df_for_eval, sim_val], ignore_index=True)
    print(f'simulation data added to DEV: {len(sim_val)} samples')

print(f"Final dev class ratio: \n{val_df_for_eval['label'].value_counts()}")

FEATURE_DIR = DATA_DIR / 'videomae_features'

# 1. 학습/검증 세트 준비
train_dataset = MultiViewDataset(train_df_copy, str(DATA_DIR / 'train'), train_transform, is_test=False, feature_dir=FEATURE_DIR)
val_dataset = MultiViewDataset(val_df_for_eval, str(DATA_DIR / 'dev'), test_transform, is_test=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    drop_last=True,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'), # CPU RAM -> GPU VRAM 전송 시 더 빠른 전송 경로 사용
    worker_init_fn=seed_worker,
    generator=make_generator(CFG['SEED'])
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'), # CPU RAM -> GPU VRAM 전송 시 더 빠른 전송 경로 사용
    worker_init_fn=seed_worker,
    generator=make_generator(CFG['SEED'] + 1)
)

# 2. 테스트 세트 준비
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')
test_df_for_infer = test_df.copy()
test_df_for_infer['sample_dir'] = str(DATA_DIR / 'test')

test_dataset = MultiViewDataset(test_df_for_infer, str(DATA_DIR / 'test'), test_transform, is_test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
    worker_init_fn=seed_worker,
    generator=make_generator(CFG['SEED'] + 2)
)

simulation data added to DEV: 96 samples
Final dev class ratio: 
label
unstable    100
stable       96
Name: count, dtype: int64


In [15]:
print(f"Sample ID: {train_df_copy.iloc[15]['id']}")

Sample ID: TRAIN_0276


In [16]:
sample = train_dataset[15]

# 현재 Dataset의 return이 (views, label, feature) 구조인지 확인
views, label, feature = sample

print("--- Data Sample Check ---")
print(f"1. Views (Image List): {len(views)} images")
print(f"   - Front View Shape: {views[0].shape}") # Tensor일 경우 [C, H, W]
print(f"   - Top View Shape:   {views[1].shape}")

print(f"2. Label: {label} (0: stable, 1: unstable)")

print(f"3. Feature (VideoMAE): {feature.shape}")
print(f"   - Sample Values: {feature[:768]}...") # 앞부분 5개 값만 확인

--- Data Sample Check ---
1. Views (Image List): 2 images
   - Front View Shape: torch.Size([3, 320, 320])
   - Top View Shape:   torch.Size([3, 320, 320])
2. Label: 0 (0: stable, 1: unstable)
3. Feature (VideoMAE): torch.Size([768])
   - Sample Values: tensor([-1.5629e+01,  2.6782e-01, -4.2127e-01, -9.9578e+00,  8.8312e-01,
         5.9123e-01,  1.8995e+00,  3.8006e-01,  6.1140e+00,  2.8737e+00,
        -1.0203e-01,  2.6179e+00, -3.3535e+00,  1.1441e+01, -1.1776e+01,
         1.1592e+00,  1.0655e+00, -6.0249e+00,  2.4530e+00,  1.2787e+00,
         2.9831e+00,  1.1960e+00, -1.9699e+00,  2.8895e-01,  1.2591e+00,
        -7.8111e+00,  1.3572e+00, -1.5601e+00, -1.6386e+00, -1.5789e+00,
         3.0715e+00,  1.1739e+01,  6.4113e+00, -1.5740e+00,  4.0037e-01,
        -3.2625e+00,  1.6838e+00,  1.6064e-01,  3.2654e+00, -1.6918e+00,
        -5.9635e-02,  1.2322e+00, -6.7701e-01,  1.4970e+00,  2.2093e+00,
         1.2282e+00,  1.2430e+00, -1.6950e+00, -3.2514e-01,  6.9369e-01,
         1.0179e

## 5. 모델 정의 (Multi-View )

In [17]:
from models import (
    EMAConfig,
    ModelEMA,
)
import dataclasses

EMA_CONFIG = EMAConfig(decay=CFG['EMA_DECAY'])
artifacts = allocate_output_paths(experiment_name='video_regularization', major_version='v3.1')

In [18]:
from dataclasses import dataclass

@dataclass(frozen=True)
class FlexibleMultiViewFeatureFusionConfig:
    backbone_name: str = "efficientnetv2_rw_s"
    pretrained: bool = True
    hidden_dim: int = 512
    mid_dim: int = 128
    dropout: float = 0.3
    out_dim: int = 1
    img_size: int = 320

class FlexibleMultiViewFeatureFusion(nn.Module):
    def __init__(self, config: FlexibleMultiViewFeatureFusionConfig):
        super().__init__()
        self.config = config

        model_kwargs = {
                "model_name": config.backbone_name,
                "pretrained": config.pretrained,
                "num_classes": 0,
                "global_pool": ""
            }
        
        if any(k in config.backbone_name.lower() for k in ["swin"]):
            model_kwargs["img_size"] = config.img_size

        self.backbone = timm.create_model(**model_kwargs)
        self.feat_dim = self.backbone.num_features

        self.fusion = nn.Sequential(
            nn.Linear(self.feat_dim * 4, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim), # 백본마다 값의 스케일이 다르므로 BN 추가 강력 권장
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.mid_dim),
            nn.ReLU()
        )
        
        # distillation용 projection: student feature → 768dim (VideoMAE와 맞춤)
        self.distill_proj = nn.Linear(config.mid_dim, 768)
        self.head = nn.Linear(config.mid_dim, config.out_dim)

    def _extract_and_pool(self, x: torch.Tensor) -> torch.Tensor:
        """
        입력된 텐서의 형태(CNN, Swin, ViT)를 동적으로 파악하여
        (Batch, Feature_Dim) 크기의 1D 벡터로 Global Average Pooling을 수행합니다.
        """
        feat = self.backbone.forward_features(x)

        # 1. 4D Tensor (CNN 또는 Swin)
        if feat.ndim == 4:
            # (B, C, H, W) 형태인 경우 (EfficientNet, ConvNeXt 등)
            if feat.shape[1] == self.feat_dim:
                return feat.mean(dim=(2, 3))
            
            # (B, H, W, C) 형태인 경우 (특정 버전의 Swin 등)
            elif feat.shape[-1] == self.feat_dim:
                return feat.mean(dim=(1, 2))

        # 2. 3D Tensor (ViT 등 Sequence 형태: (B, Sequence_Length, C))
        elif feat.ndim == 3:
            if feat.shape[-1] == self.feat_dim:
                return feat.mean(dim=1) # 모든 토큰의 평균을 냄 (GAP)

        raise ValueError(f"Unsupported feature shape: {tuple(feat.shape)} for backbone={self.config.backbone_name}")

    def forward(self, views, return_feat=False):
        # 2. 백본을 거쳐 무조건 (B, feat_dim) 형태로 맞춰진 특징 벡터 획득
        f1 = self._extract_and_pool(views[0])
        f2 = self._extract_and_pool(views[1])

        combined = torch.cat([f1, f2, torch.abs(f1 - f2), f1 * f2], dim=1)
        
        # 3. 차원에 구애받지 않고 안전하게 Concat 진행
        fused = self.fusion(combined)  # (B, mid_dim)
        out   = self.head(fused)                         # (B, 1)

        if return_feat:
            student_feat = self.distill_proj(fused)      # (B, 768)
            return out, student_feat

        return out

## 6. 학습 및 검증 루프 구현

In [19]:
from torch.cuda.amp import autocast, GradScaler

def train_one_epoch(model, loader, criterion, optimizer, device, ema=None, distill_weight=0.1, scaler=None, epoch=0):
    model.train()
    
    # 에포크 전체 Loss를 누적할 변수
    epoch_total_loss = 0
    epoch_task_loss = 0
    epoch_distill_loss = 0
    
    pbar = tqdm(enumerate(loader), total=len(loader), desc=f"Epoch {epoch}", dynamic_ncols=True, ascii=True)
    
    for i, (views, labels, teacher_feats) in pbar:
        views         = [v.to(device) for v in views]
        labels        = labels.to(device).float()
        teacher_feats = teacher_feats.to(device)

        optimizer.zero_grad()

        with autocast():
            # 1. Task Loss
            outputs, student_feats = model(views, return_feat=True)
            outputs = outputs.view(-1)
            loss_task = criterion(outputs, labels)

            # 2. Distillation Loss
            valid_mask = teacher_feats.abs().sum(dim=1) > 0 
            if valid_mask.any():
                loss_distill_raw = F.mse_loss(
                    student_feats[valid_mask],
                    teacher_feats[valid_mask].detach()
                )
            else:
                loss_distill_raw = torch.tensor(0.0, device=device)

            weighted_distill = distill_weight * loss_distill_raw
            loss = loss_task + weighted_distill

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if ema is not None:
            ema.update(model)

        # 값 누적 (평균 계산용)
        epoch_total_loss += loss.item()
        epoch_task_loss += loss_task.item()
        epoch_distill_loss += weighted_distill.item()

    # 에포크 종료 후 평균값 계산
    num_batches = len(loader)
    avg_total = epoch_total_loss / num_batches
    avg_task = epoch_task_loss / num_batches
    avg_distill = epoch_distill_loss / num_batches

    # [최종 출력] 에포크 결과 요약 프린트
    print(f"\n>> [Epoch {epoch} Summary] | Total Loss:   {avg_total:.6f} |   Task Loss:    {avg_task:.6f} | Distill Loss: {avg_distill:.6f} | Learning Rate: {optimizer.param_groups[0]['lr']:.8f}")

    # WandB에는 에포크 단위로 기록
    wandb.log({
        'train/epoch_total_loss': avg_total,
        'train/epoch_task_loss': avg_task,
        'train/epoch_distill_loss': avg_distill,
        'epoch': epoch
    })

    return avg_total

def validate(model, loader, criterion, device):
    model.eval()
    val_loss  = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation", dynamic_ncols=True, ascii=True):
            views, labels = batch[0], batch[1]
            views  = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            loss    = criterion(outputs, labels)
            val_loss += loss.item()

            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_preds,  axis=0).astype(np.float64)
    all_labels = np.concatenate(all_labels, axis=0).astype(np.float64)

    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score     = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

# -------------------------
# TTA helpers
# -------------------------
def _center_crop_and_resize(x, crop_ratio=0.95):
    # x: [B, C, H, W]
    b, c, h, w = x.shape
    ch, cw = int(h * crop_ratio), int(w * crop_ratio)
    y1 = (h - ch) // 2
    x1 = (w - cw) // 2
    cropped = x[:, :, y1:y1 + ch, x1:x1 + cw]
    return F.interpolate(cropped, size=(h, w), mode='bilinear', align_corners=False)


def apply_tta_to_views(views, tta_name):
    if tta_name == 'none':
        return views
    if tta_name == 'hflip':
        return [torch.flip(v, dims=[3]) for v in views]
    if tta_name == 'crop95':
        return [_center_crop_and_resize(v, crop_ratio=0.95) for v in views]
    raise ValueError(f'Unknown TTA: {tta_name}')


def predict_probs_with_tta(model, loader, device, tta_names=None, has_labels=False, desc='Inference TTA'):
    if tta_names is None:
        tta_names = ['none']

    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, dynamic_ncols=True, ascii=True):
            if has_labels:
                views, labels = batch[0], batch[1]
                labels = labels.to(device).float()
                all_labels.extend(labels.cpu().numpy())
            else:
                views = batch[0]  # ← [views] → views

            views = [v.to(device) for v in views]

            probs_sum = None
            for tta_name in tta_names:
                tta_views = apply_tta_to_views(views, tta_name)
                logits = model(tta_views).view(-1)
                probs  = torch.sigmoid(logits)
                probs_sum = probs if probs_sum is None else (probs_sum + probs)

            probs_avg = probs_sum / len(tta_names)
            all_probs.extend(probs_avg.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    if has_labels:
        return all_probs, np.array(all_labels, dtype=np.float64)
    return all_probs


def evaluate_tta_on_dev(model, loader, device, tta_candidates):
    rows = []
    for tta_names in tta_candidates:
        probs, labels = predict_probs_with_tta(
            model, loader, device, tta_names=tta_names, has_labels=True,
            desc=f'Dev TTA {tta_names}'
        )

        eps = 1e-15
        p = np.clip(probs, eps, 1 - eps)
        logloss_score = -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
        acc_score = np.mean((probs > 0.5) == labels)

        rows.append({
            'tta_names': tta_names,
            'n_tta': len(tta_names),
            'val_logloss': float(logloss_score),
            'val_acc': float(acc_score),
        })

    return pd.DataFrame(rows).sort_values('val_logloss', ascending=True).reset_index(drop=True)

In [20]:
distill_weight_list = CFG['DISTILL_WEIGHT']
backbone_model_list = BACKBONE_CANDIDATES

print(distill_weight_list)
print(backbone_model_list)

[0.0001, 0.001, 0.01, 0.1]
['efficientnetv2_rw_s']


In [21]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

def train_single_backbone(backbone_name : str) :
    safe_name = backbone_name.replace("/", "_").replace(".", "_")
    sweep_results = []
    distill_weight_list = CFG['DISTILL_WEIGHT']
    target_epochs = CFG['EPOCHS']

    for distill_weight in distill_weight_list :
        best_model_path = WEIGHT_DIR / f"best_model_{safe_name}_dw_{distill_weight}.pt"
        submission_path = SUBMISSION_DIR / f"submission_{safe_name}_dw_{distill_weight}.csv"

        if best_model_path.exists():
            print(f"\n[Skip] 이미 학습된 모델이 존재합니다: {best_model_path.name}")
            checkpoint = torch.load(best_model_path, map_location='cpu', weights_only=False)
            sweep_results.append({
                "backbone": backbone_name,
                "target_epochs": target_epochs,
                "distill_weight": distill_weight,
                "best_epoch": checkpoint.get('epoch', -1),
                "val_logloss": checkpoint.get('val_logloss', -1),
                "val_acc": checkpoint.get('val_acc', -1),
                "model_path": str(best_model_path),
            })
            continue

        print(f"\n=======================================================")
        print(f" Starting Sweep: {backbone_name} | distill_weight = {distill_weight}")
        print(f"=======================================================\n")

        run_name = f"{safe_name}_dw{distill_weight}"
        run_config = CFG.copy()
        run_config['CURRENT_DISTILL_WEIGHT'] = distill_weight
        
        wandb.init(
            project="structure-stability", 
            name=f"{run_name}",
            group=EXPERIMENT_NAME,
            config=run_config,
            reinit=True # 중요: 이전 Run을 닫고 새로 시작
        )

        model_config = FlexibleMultiViewFeatureFusionConfig(backbone_name = backbone_name)
        print(model_config)
        model = FlexibleMultiViewFeatureFusion(model_config).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR']
        )
        ema = ModelEMA(model, EMA_CONFIG)

        print(f"Artifact version: {artifacts['version']}")
        print(f"Regularization -> weight_decay={CFG['WEIGHT_DECAY']}")
        print(f"Scheduler -> CosineAnnealingLR(T_max={target_epochs}, eta_min={CFG['MIN_LR']})")
        print(f"EMA -> decay={CFG['EMA_DECAY']}, use_for_eval={CFG['EMA_USE_FOR_EVAL']}")
        print(f"Early Stopping -> Patience={CFG['PATIENCE']}")
        print(f"Distill Weight -> decay={distill_weight}")

        # --- Init ---
        best_epoch = -1
        early_stop_counter = 0 
        best_logloss = float('inf')
        scaler = GradScaler()

         # --- Main Loop ---
        for epoch in range(1, CFG['EPOCHS'] + 1):
            avg_train_loss = train_one_epoch(
                model, train_loader, criterion, optimizer, device,
                ema=ema,
                distill_weight=distill_weight,
                scaler=scaler,
                epoch = epoch  
            )
            
            eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
            val_logloss, val_acc = validate(eval_model, val_loader, criterion, device)

            # WandB 기록
            wandb.log({
                'epoch'          : epoch,
                'val/logloss'    : val_logloss,
                'val/acc'        : val_acc,
                'lr'             : optimizer.param_groups[0]['lr'],
                'best_logloss'   : best_logloss,
                'early_stop_count': early_stop_counter
            }, step=epoch)

            # --- Early Stopping 및 모델 저장 로직 ---
            if val_logloss < best_logloss:
                # 성능 개선됨
                best_logloss = val_logloss
                best_epoch = epoch
                early_stop_counter = 0  # 개선되었으므로 카운터 초기화
                
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'ema_state_dict': ema.ema_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_logloss': val_logloss,
                    'val_acc': val_acc,
                    'cfg': CFG,
                }, best_model_path)
                print(f"  -> Best model saved: {best_model_path} (epoch={epoch}, val_logloss={val_logloss:.6f})")
            else:
                # 성능 개선 없음
                early_stop_counter += 1
                print(f"  -> No improvement. EarlyStopping counter: {early_stop_counter}/{CFG['PATIENCE']}")

            # 조기 종료 체크
            if early_stop_counter >= CFG['PATIENCE']:
                print(f" \n! [Early Stopping] Training stopped at epoch {epoch}. Best Epoch was {best_epoch}.")
                break

            scheduler.step()
            current_lr = optimizer.param_groups[0]['lr']

            print(f"Epoch [{epoch}]")
            print(f"  - LR: {current_lr:.8f}")
            print(f"  - Train Loss: {avg_train_loss:.4f}")
            print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

        wandb.finish()

        # 학습 종료 후 best 가중치 로드
        if best_model_path.exists():
            checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
            sweep_results.append({
                "backbone": backbone_name,
                "target_epochs": target_epochs,
                "distill_weight": distill_weight,
                "best_epoch": checkpoint['epoch'],
                "val_logloss": checkpoint['val_logloss'],
                "val_acc": checkpoint['val_acc'],
                "model_path": str(best_model_path)
            })

    return sweep_results

In [22]:
import pandas as pd

results = []

for backbone_name in selected_backbones:
    try:
        result_list = train_single_backbone(backbone_name)
        results.extend(result_list) 
        
    except Exception as exc:
        print(f"FAILED: {backbone_name} | Error: {exc}")

print(results)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vsc/.netrc.



 Starting Sweep: efficientnetv2_rw_s | distill_weight = 0.0001



wandb: Currently logged in as: jungseonglian (uailab-unist_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


FlexibleMultiViewFeatureFusionConfig(backbone_name='efficientnetv2_rw_s', pretrained=True, hidden_dim=512, mid_dim=128, dropout=0.3, out_dim=1, img_size=320)
Artifact version: v3.1.0
Regularization -> weight_decay=0.0001
Scheduler -> CosineAnnealingLR(T_max=30, eta_min=1e-06)
EMA -> decay=0.999, use_for_eval=True
Early Stopping -> Patience=10
Distill Weight -> decay=0.0001


/tmp/ipykernel_3957266/3947437432.py:64: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/86 [00:00<?, ?it/s]/tmp/ipykernel_3957266/476825525.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1: 100%|##########| 86/86 [00:30<00:00,  2.84it/s]


...


>> [Epoch 30 Summary] | Total Loss:   0.083939 |   Task Loss:    0.000937 | Distill Loss: 0.083002 | Learning Rate: 0.00000182


Validation: 100%|##########| 12/12 [00:09<00:00,  1.20it/s]


  -> Best model saved: /home/vsc/LLM_TUNE/structure-stability/outputs/weights/Data_AUG_Test/v3.1/best_model_efficientnetv2_rw_s_dw_0.1.pt (epoch=30, val_logloss=0.093255)
Epoch [30]
  - LR: 0.00000100
  - Train Loss: 0.0839
  - Val Log-Loss: 0.093255 | Val Acc: 0.9688


best_logloss,████▇▇▆▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
early_stop_count,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
lr,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/epoch_distill_loss,█▆▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_task_loss,█▄▅▄▃▃▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_total_loss,█▆▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▄▆▅▅▆▆▆▇▇▇▇▇▇▇██████████████
val/logloss,████▇▇▆▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
best_logloss,0.09417
early_stop_count,0


[{'backbone': 'efficientnetv2_rw_s', 'target_epochs': 30, 'distill_weight': 0.0001, 'best_epoch': 30, 'val_logloss': np.float64(0.24179851089127413), 'val_acc': np.float64(0.90625), 'model_path': '/home/vsc/LLM_TUNE/structure-stability/outputs/weights/Data_AUG_Test/v3.1/best_model_efficientnetv2_rw_s_dw_0.0001.pt'}, {'backbone': 'efficientnetv2_rw_s', 'target_epochs': 30, 'distill_weight': 0.001, 'best_epoch': 30, 'val_logloss': np.float64(0.1197422554198681), 'val_acc': np.float64(0.9583333333333334), 'model_path': '/home/vsc/LLM_TUNE/structure-stability/outputs/weights/Data_AUG_Test/v3.1/best_model_efficientnetv2_rw_s_dw_0.001.pt'}, {'backbone': 'efficientnetv2_rw_s', 'target_epochs': 30, 'distill_weight': 0.01, 'best_epoch': 12, 'val_logloss': np.float64(0.318118210451541), 'val_acc': np.float64(0.890625), 'model_path': '/home/vsc/LLM_TUNE/structure-stability/outputs/weights/Data_AUG_Test/v3.1/best_model_efficientnetv2_rw_s_dw_0.01.pt'}, {'backbone': 'efficientnetv2_rw_s', 'target_e

In [23]:
def run_tta_and_inference(
    model, 
    best_model_path, 
    submission_path, 
    val_loader, 
    test_loader, 
    test_df, 
    device, 
    cfg
):
    print(f"\n=======================================================")
    print(f"🔍 Starting TTA & Inference Phase")
    print(f"=======================================================\n")
    
    if not best_model_path.exists():
        print(f"❌ Error: Best model weights not found at {best_model_path}")
        return None, None

    # 1. 가중치 로드
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
    
    # EMA 사용 여부에 따라 적절한 모델의 가중치를 불러옵니다.
    if cfg.get('EMA_USE_FOR_EVAL', False) and 'ema_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['ema_state_dict'])
        print(" -> Loaded EMA weights for inference.")
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(" -> Loaded standard weights for inference.")
        
    model.eval()

    # 2. dev(Val)셋에서 TTA 조합 성능 비교
    candidate_ttas = cfg['TTA_CANDIDATES']
    tta_result_df = evaluate_tta_on_dev(model, val_loader, device, candidate_ttas)
    display(tta_result_df) # Jupyter 환경용 표 출력

    # 가장 성능이 좋은 TTA 조합 선택
    best_tta_names = tta_result_df.iloc[0]['tta_names']
    best_tta_logloss = tta_result_df.iloc[0]['val_logloss']
    print(f"🏆 Best TTA on dev: {best_tta_names} (LogLoss: {best_tta_logloss:.6f})")

    # 3. Best TTA로 Test 추론
    all_probs = predict_probs_with_tta(
        model, test_loader, device,
        tta_names=best_tta_names,
        has_labels=False,
        desc=f'Inference with TTA ({best_tta_names})'
    )

    # 4. Submission CSV 생성 및 저장
    submission = pd.DataFrame({
        'id': test_df['id'],
        'unstable_prob': all_probs,
        'stable_prob': 1.0 - all_probs
    })

    submission.to_csv(submission_path, encoding='UTF-8-sig', index=False)
    print(f'✅ Submission saved to: {submission_path}')
    
    return best_tta_names, best_tta_logloss

In [26]:
import pandas as pd
from pathlib import Path

results = []

for backbone_name in BACKBONE_CANDIDATES:
    try:
        # ==========================================================
        # 단계 1: 학습된 결과물을 바탕으로 TTA 검증 및 최종 추론
        # ==========================================================
        result_list = train_single_backbone(backbone_name)
        print(f"example || {result_list[0]}")

        for res in result_list:
            best_model_path = Path(res["model_path"])
            distill_weight = res["distill_weight"]
            safe_name = backbone_name.replace("/", "_").replace(".", "_")
            submission_path = SUBMISSION_DIR / f"submission_{safe_name}_dw_{distill_weight}.csv"
            
            # 모델 구조 재초기화 (빈 껍데기 준비)
            model_config = FlexibleMultiViewFeatureFusionConfig(
                backbone_name=backbone_name,
                img_size=CFG['IMG_SIZE'] # 앞서 추가했던 img_size 유지
            )
            model = FlexibleMultiViewFeatureFusion(model_config).to(device)
            
            # 별도로 분리한 추론 함수 실행
            best_tta, tta_logloss = run_tta_and_inference(
                model=model,
                best_model_path=best_model_path,
                submission_path=submission_path,
                val_loader=val_loader,
                test_loader=test_loader,
                test_df=test_df,
                device=device,
                cfg=CFG
            )
            
            # 최종 결과 딕셔너리에 추론 결과 업데이트
            if best_tta is not None:
                res["best_tta"] = str(best_tta)
                res["tta_val_logloss"] = tta_logloss
                res["submission_path"] = str(submission_path)
                
        # 모든 처리가 끝난 결과를 리스트에 누적
        results.extend(result_list) 
        
    except Exception as exc:
        print(f"FAILED: {backbone_name} | Error: {exc}")

# (선택) 전체 백본 실험이 끝난 후 최종 요약 표 출력
final_results_df = pd.DataFrame(results)
display(final_results_df)


[Skip] 이미 학습된 모델이 존재합니다: best_model_efficientnetv2_rw_s_dw_0.0001.pt

[Skip] 이미 학습된 모델이 존재합니다: best_model_efficientnetv2_rw_s_dw_0.001.pt

[Skip] 이미 학습된 모델이 존재합니다: best_model_efficientnetv2_rw_s_dw_0.01.pt

[Skip] 이미 학습된 모델이 존재합니다: best_model_efficientnetv2_rw_s_dw_0.1.pt
example || {'backbone': 'efficientnetv2_rw_s', 'target_epochs': 30, 'distill_weight': 0.0001, 'best_epoch': 30, 'val_logloss': np.float64(0.24179851089127413), 'val_acc': np.float64(0.90625), 'model_path': '/home/vsc/LLM_TUNE/structure-stability/outputs/weights/Data_AUG_Test/v3.1/best_model_efficientnetv2_rw_s_dw_0.0001.pt'}

🔍 Starting TTA & Inference Phase

 -> Loaded EMA weights for inference.


Dev TTA ['none', 'hflip', 'crop95']: 100%|##########| 13/13 [00:13<00:00,  1.02s/it]


,tta_names,n_tta,val_logloss,val_acc
0,"[none, hflip, crop95]",3,0.220818,0.908163
1,"[none, hflip]",2,0.220896,0.903061
2,[none],1,0.237740,0.908163


🏆 Best TTA on dev: ['none', 'hflip', 'crop95'] (LogLoss: 0.220818)


Inference with TTA (['none', 'hflip', 'crop95']): 100%|##########| 63/63 [00:33<00:00,  1.90it/s]


...

,tta_names,n_tta,val_logloss,val_acc
0,"[none, hflip]",2,0.055659,0.969388
1,"[none, hflip, crop95]",3,0.057486,0.969388
2,[none],1,0.091766,0.969388


🏆 Best TTA on dev: ['none', 'hflip'] (LogLoss: 0.055659)


Inference with TTA (['none', 'hflip']): 100%|##########| 63/63 [00:22<00:00,  2.80it/s]

✅ Submission saved to: /home/vsc/LLM_TUNE/structure-stability/outputs/submissions/Data_AUG_Test/v3.1/submission_efficientnetv2_rw_s_dw_0.1.csv


,backbone,target_epochs,distill_weight,best_epoch,val_logloss,val_acc,model_path,best_tta,tta_val_logloss,submission_path
0,efficientnetv2_rw_s,30,0.0001,30,0.241799,0.906250,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip', 'crop95']",0.220818,/home/vsc/LLM_TUNE/structure-stability/outputs...
1,efficientnetv2_rw_s,30,0.0010,30,0.119742,0.958333,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip']",0.105379,/home/vsc/LLM_TUNE/structure-stability/outputs...
2,efficientnetv2_rw_s,30,0.0100,12,0.318118,0.890625,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip', 'crop95']",0.310624,/home/vsc/LLM_TUNE/structure-stability/outputs...
3,efficientnetv2_rw_s,30,0.1000,30,0.093255,0.968750,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip']",0.055659,/home/vsc/LLM_TUNE/structure-stability/outputs...


In [27]:
result_df = pd.DataFrame(results)
if not result_df.empty:
    result_df = result_df.sort_values(
        by=["val_logloss", "val_acc"], 
        ascending=[True, False]
    ).reset_index(drop=True)

    # 출력 및 저장
    display(result_df)
    
    # 저장할 디렉토리가 없다면 생성
    os.makedirs(EDA_DIR, exist_ok=True)
    result_df.to_csv(EDA_DIR / "mv_caf_ensemble_v1.0_eval.csv", index=False)
    print("✨ 모든 스윕 완료 및 결과 저장 성공!")
else:
    print("결과가 비어 있습니다. 학습 중 에러가 발생했는지 확인해주세요.")

,backbone,target_epochs,distill_weight,best_epoch,val_logloss,val_acc,model_path,best_tta,tta_val_logloss,submission_path
0,efficientnetv2_rw_s,30,0.1000,30,0.093255,0.968750,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip']",0.055659,/home/vsc/LLM_TUNE/structure-stability/outputs...
1,efficientnetv2_rw_s,30,0.0010,30,0.119742,0.958333,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip']",0.105379,/home/vsc/LLM_TUNE/structure-stability/outputs...
2,efficientnetv2_rw_s,30,0.0001,30,0.241799,0.906250,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip', 'crop95']",0.220818,/home/vsc/LLM_TUNE/structure-stability/outputs...
3,efficientnetv2_rw_s,30,0.0100,12,0.318118,0.890625,/home/vsc/LLM_TUNE/structure-stability/outputs...,"['none', 'hflip', 'crop95']",0.310624,/home/vsc/LLM_TUNE/structure-stability/outputs...


✨ 모든 스윕 완료 및 결과 저장 성공!
